In [ ]:
!pip install torch==2.6.0+cu124 --extra-index-url https://download.pytorch.org/whl/cu124
!pip install -r requirements.txt
!pip install sagemaker
!pip install dotenv

In [10]:
from dotenv import load_dotenv
from sagemaker import Model
from sagemaker.predictor import Predictor
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer
import boto3
import os
import sys
sys.path.append('..')
load_dotenv()

ACCOUNT_ID = os.getenv("AWS_ACCOUNT_ID")
ROLE_ARN = os.getenv("AWS_ROLE_ARN")
REGION = os.getenv("REGION")
ENTRYPOINT_REP_NAME = os.getenv("ENTRYPOINT_REP_NAME")

In [ ]:
model = Model(
    image_uri=f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{ENTRYPOINT_REP_NAME}:latest",
    role=f"arn:aws:iam::{ROLE_ARN}",
    name="ensemble-model"
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.g5.4xlarge",
    serializer=JSONSerializer(),
    deserializer=JSONDeserializer()
)

In [ ]:
sagemaker_client = boto3.client("sagemaker")

new_endpoint_config_name = "Ensemble-endpoint-config"
model_name = "Binding-prediction-sEH-BRD4-HSA"
instance_type = "ml.g5.8xlarge"
initial_instance_count = 1

# Create the new endpoint configuration
sagemaker_client.create_endpoint_config(
    EndpointConfigName=new_endpoint_config_name,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": 'Ensemble',
            "InstanceType": instance_type,
            "InitialInstanceCount": initial_instance_count,
        }
    ],
)

print(f"Created new endpoint configuration: {new_endpoint_config_name}")

In [ ]:
endpoint_name = "Ensemble-endpoint"

# Create the endpoint
sagemaker_client.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=new_endpoint_config_name,
)

print(f"Created new endpoint: {endpoint_name}")